# First questions — the analytics shadow

`reports/analytics.duckdb` is your vault's ledger as a real SQL database — every posting, price, and daily balance snapshot, rebuilt whole by `tools/run reports.py` (or `sara-analytics`). Two rules before anything else:

- **Truth vs shadow.** The beancount ledger is the only source of truth. This file is a disposable cache — never edit it, never archive it, and regenerate it whenever it looks stale (`build_info` below says when and from which vault commit it was built).
- **Read-only etiquette.** Always connect with `read_only=True`. DuckDB is single-writer: a stray read-write handle blocks the next rebuild.

Notebook deps: `pip install duckdb pandas matplotlib`. Point `FINANCE_VAULT` at any vault — the demo vault (`init_vault.sh --demo /tmp/demo-vault`) is perfect for a test drive, and this notebook assumes nothing about your real data.

**Running it:** point the notebook kernel at the vault's venv (`$VAULT/.venv/bin/python` — it already has duckdb + pandas), and `$VAULT/.venv/bin/pip install matplotlib` once for the charts. Set `FINANCE_VAULT` if the vault isn't `~/Finance`.

In [ ]:
import os
from pathlib import Path

import duckdb

vault = Path(os.environ.get("FINANCE_VAULT", str(Path.home() / "Finance")))
con = duckdb.connect(str(vault / "reports" / "analytics.duckdb"), read_only=True)
home = con.sql("SELECT home_currency FROM build_info").fetchone()[0]
con.sql("SELECT * FROM build_info").df()

## 1. What's the monthly spend trend, by category?

`monthly_flows` is the postings fact table pre-bucketed by month and account; expense categories are the second account segment (`Expenses:Food:...` → `Food`).

In [ ]:
import matplotlib.pyplot as plt

PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]  # fixed slot order

spend = con.sql("""
    SELECT month, split_part(account, ':', 2) AS category, sum(total_home) AS spent
    FROM monthly_flows
    WHERE root = 'Expenses'
    GROUP BY ALL
    ORDER BY month
""").df()

top = spend.groupby("category")["spent"].sum().nlargest(5).index.tolist()
pivot = (spend[spend.category.isin(top)]
         .pivot_table(index="month", columns="category", values="spent")
         .fillna(0)[top])

fig, ax = plt.subplots(figsize=(9, 4.5))
for slot, category in enumerate(top):
    ax.plot(pivot.index, pivot[category], color=PALETTE[slot], linewidth=2, label=category)
ax.set_title("Monthly spend — top 5 categories")
ax.set_ylabel(f"spend ({home})")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.margins(x=0.01)
fig.tight_layout()

## 2. What does the net-worth curve look like?

`balances_daily` values every account per day at the latest known price; the `net_worth_daily` view rolls it up. `liquid_home` additionally excludes any commodities `rules.toml` marks illiquid — the household's spendable figure.

In [ ]:
nw = con.sql("SELECT date, net_worth_home, liquid_home FROM net_worth_daily").df()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(nw["date"], nw["net_worth_home"], color=PALETTE[0], linewidth=2)
ax.fill_between(nw["date"], nw["net_worth_home"], color=PALETTE[0], alpha=0.12)
ax.set_title("Net worth, daily")
ax.set_ylabel(home)
ax.spines[["top", "right"]].set_visible(False)
ax.margins(x=0.01)
fig.tight_layout()

nw.tail(3)

## 3. Where did the money actually go, last 90 days?

Straight off the `postings` fact table — payee plus `amount_home`, expenses only, windowed to the last 90 days of ledger activity.

In [ ]:
con.sql("""
    SELECT payee,
           count(DISTINCT txn_id) AS txns,
           sum(amount_home) AS spent
    FROM postings
    WHERE split_part(account, ':', 1) = 'Expenses'
      AND payee IS NOT NULL
      AND date >= (SELECT max(date) FROM postings) - INTERVAL 90 DAY
    GROUP BY payee
    ORDER BY spent DESC
    LIMIT 10
""").df()

Done — release the read handle so the next `tools/run reports.py` can swap in a fresh build. From here it's ordinary SQL: `register` has running balances per account, `prices` the full price history, `accounts` the owner/institution dims, and the parquet twins in `reports/exports/` load anywhere (`pl.read_parquet`, `pd.read_parquet`, Spark...) with no DuckDB at all.

In [ ]:
con.close()